In [9]:
# imports 
import os
import re
from pathlib import Path
import yaml

# Data & math
import pandas as pd
import numpy as np

# Viz
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Notebook viz defaults
sns.set(context='notebook', style='whitegrid')
pio.renderers.default = 'notebook_connected' 

print(plt.style.available)
plt.style.use('fivethirtyeight')
plt.rcParams['figure.figsize'] = (8, 4)


print('✅ Imports ready')

['Solarize_Light2', '_classic_test_patch', '_mpl-gallery', '_mpl-gallery-nogrid', 'bmh', 'classic', 'dark_background', 'fast', 'fivethirtyeight', 'ggplot', 'grayscale', 'petroff10', 'seaborn-v0_8', 'seaborn-v0_8-bright', 'seaborn-v0_8-colorblind', 'seaborn-v0_8-dark', 'seaborn-v0_8-dark-palette', 'seaborn-v0_8-darkgrid', 'seaborn-v0_8-deep', 'seaborn-v0_8-muted', 'seaborn-v0_8-notebook', 'seaborn-v0_8-paper', 'seaborn-v0_8-pastel', 'seaborn-v0_8-poster', 'seaborn-v0_8-talk', 'seaborn-v0_8-ticks', 'seaborn-v0_8-white', 'seaborn-v0_8-whitegrid', 'tableau-colorblind10']
✅ Imports ready


In [10]:
# locate the repo root
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / 'project_config.yaml').exists():
            return p
    return start

ROOT = find_repo_root()
print('Repo root:', ROOT)

# Load config 
CFG_PATH = ROOT / 'project_config.yaml'
if CFG_PATH.exists():
    with open(CFG_PATH, 'r') as f:
        cfg = yaml.safe_load(f)
else:
    cfg = {}

# ---- ATHLETE / FPS defaults ----
ATHLETE = cfg.get('athlete', 'athlete_01')  
FPS = int(cfg.get('player_tracking_fps', 30))

DATA_DIR = ROOT / 'data' / ATHLETE
assert DATA_DIR.exists(), f'DATA_DIR not found: {DATA_DIR}'

# Discover sessions for this athlete (folders under data/<ATHLETE>)
SESSIONS = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])

# Create boolean flags per session and a dict to track inclusion
SESSION_INCLUDE = {}
for s in SESSIONS:
    var = 'include_' + re.sub(r'[^0-9a-zA-Z_]', '_', s)
    globals()[var] = True  # default True
    SESSION_INCLUDE[s] = True

print('ATHLETE:', ATHLETE, 'FPS:', FPS)
print('Sessions discovered:', SESSIONS)
print('Per-session booleans created in globals():', [k for k in globals() if k.startswith('include_')])

def selected_sessions():
    # Returns the list of sessions with their boolean 'include_*' variable set to True.
    out = []
    for s in SESSIONS:
        var = 'include_' + re.sub(r'[^0-9a-zA-Z_]', '_', s)
        if globals().get(var, False):
            out.append(s)
    return out

print('Initially selected sessions:', selected_sessions())


Repo root: /Users/kwill55/RVL/personalized-sports-optimization
ATHLETE: tyler_haws FPS: 60
Sessions discovered: ['session_with_tyler_haws_gym']
Per-session booleans created in globals(): ['include_session_with_tyler_haws_gym']
Initially selected sessions: ['session_with_tyler_haws_gym']


In [11]:
# definitions for gathering angles

# TODO add load_angles_wide later on 

def list_angle_csvs(athlete: str, sessions: list[str], root: Path = ROOT) -> pd.DataFrame:
    # Return a DataFrame: ['session', 'clip', 'path'] for all *_angles.csv under data/<ATHLETE>/<SESSION>/metrics/3d_angles/
    rows = []
    base = root / 'data' / athlete
    for s in sessions:
        angles_dir = base / s / 'metrics' / '3d_angles'
        if not angles_dir.exists():
            continue
        for p in sorted(angles_dir.glob('*_angles.csv')):
            # Try to extract a clip id/number from the filename
            m = re.search(r'(\d+)', p.stem)
            clip = int(m.group(1)) if m else None
            rows.append({'session': s, 'clip': clip, 'path': p})
    return pd.DataFrame(rows)

def load_angles_long(angle_files: pd.DataFrame,
                     downcast_float32: bool = True) -> pd.DataFrame:
    """
    Read all *_angles.csv into a long/tidy table with columns:
    ['athlete','session','clip','file','frame','time_s','angle','value']
    """
    dfs = []
    for _, row in angle_files.iterrows():
        p = Path(row['path'])
        df = pd.read_csv(p)

        # ensure frame exists
        if 'frame' not in df.columns:
            df.insert(0, 'frame', np.arange(len(df), dtype=int))

        # melt wide → long
        long = df.melt(id_vars=['frame'], var_name='angle', value_name='value')

        # attach metadata
        long['athlete'] = ATHLETE
        long['session'] = row['session']
        long['clip']    = row['clip']
        long['file']    = p.name   # or p.stem if you want to drop ".csv"

        if downcast_float32:
            long['value'] = pd.to_numeric(long['value'], errors='coerce').astype('float32')

        dfs.append(long)

    if not dfs:
        return pd.DataFrame(columns=['athlete','session','clip','file','frame','time_s','angle','value'])

    out = pd.concat(dfs, ignore_index=True)

    # compute time from frame using FPS
    out['time_s'] = out['frame'] / float(max(FPS, 1))

    # order columns
    out = out[['athlete','session','clip','file','frame','time_s','angle','value']]
    return out


In [12]:
#defintions for gathering phases 

def list_phase_csvs(athlete: str, sessions: list[str], root: Path = ROOT) -> pd.DataFrame:
    """
    Discover freethrow_phases.csv under each selected session.
    Returns DataFrame with columns: ['session','path']
    """
    rows = []
    base = root / 'data' / athlete
    for s in sessions:
        session_root = base / s
        for p in session_root.rglob('freethrow_phases.csv'):
            rows.append({'session': s, 'path': p})
    return pd.DataFrame(rows)

def extract_clip_id_from_file_str(file_str: str):
    """'clip_003.mp4' -> 3, else None."""
    try:
        stem = Path(str(file_str)).stem
    except Exception:
        stem = str(file_str)
    m = re.search(r'(\d+)', stem)
    return int(m.group(1)) if m else None

def load_phases_table(phase_files: pd.DataFrame,
                      athlete: str,
                      fps: float = 60) -> pd.DataFrame:
    """
    Load all freethrow_phases.csv files into a single table with:
    ['athlete','session','clip','file','windup_start','release_frame','followthrough_end',
     'windup_t','release_t','followthrough_t'].
    """
    dfs = []
    for _, row in phase_files.iterrows():
        p = Path(row['path'])
        df = pd.read_csv(p)

        # normalize names (case-insensitive)
        cols = {c.lower(): c for c in df.columns}
        need = ['file','raw_windup_start','raw_release_frame','raw_followthrough_end']
        missing = [c for c in need if c not in cols]
        if missing:
            raise ValueError(f"Missing columns in {p}: {missing}")

        # select with original casing, then standardize names
        df = df[[cols[c] for c in need]].copy()
        df.columns = need

        # align keys with angles_long
        df['athlete'] = athlete
        df['session'] = row['session']
        df['clip']    = df['file'].apply(extract_clip_id_from_file_str)

        # times in seconds
        denom = float(max(fps, 1))
        df['windup_t']         = df['raw_windup_start']      / denom
        df['release_t']        = df['raw_release_frame']     / denom
        df['followthrough_t']  = df['raw_followthrough_end'] / denom

        # enforce numeric (avoids type-mismatch on merge)
        for c in ['raw_windup_start','raw_release_frame','raw_followthrough_end']:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')

        dfs.append(df)

    if not dfs:
        return pd.DataFrame(columns=[
            'athlete','session','clip','file',
            'raw_windup_start','raw_release_frame','raw_followthrough_end',
            'windup_t','release_t','followthrough_t'
        ])

    out = pd.concat(dfs, ignore_index=True)
    out = out.dropna(subset=['clip'])  # optional
    out = out[['athlete','session','clip','file',
               'raw_windup_start','raw_release_frame','raw_followthrough_end',
               'windup_t','release_t','followthrough_t']]
    return out


In [13]:
# create data frames for phases and angles 

#phases data frames
phase_files_df = list_phase_csvs(ATHLETE, selected_sessions())
display(phase_files_df)
print(f'Found {len(phase_files_df)} phase files across {phase_files_df['session'].nunique()} sessions.')

phases_df = load_phases_table(phase_files_df, athlete=ATHLETE, fps=FPS)
display(phases_df.head())
print(f'Loaded {len(phases_df)} phase rows across {phases_df["session"].nunique()} sessions.')


#angles data frames 
angle_files_df = list_angle_csvs(ATHLETE, selected_sessions())
display(angle_files_df.head())
print(f'Found {len(angle_files_df)} angle files across {angle_files_df['session'].nunique()} sessions.')

angles_long_df = load_angles_long(angle_files_df)
display(angles_long_df.head())
print(f'Loaded {len(angles_long_df)} angle rows across {angles_long_df["session"].nunique()} sessions.')


,session,path
0,session_with_tyler_haws_gym,/Users/kwill55/RVL/personalized-sports-optimiz...


Found 1 phase files across 1 sessions.


,athlete,session,clip,file,raw_windup_start,raw_release_frame,raw_followthrough_end,windup_t,release_t,followthrough_t
0,tyler_haws,session_with_tyler_haws_gym,2,freethrow002.avi,158,205,250,2.633333,3.416667,4.166667
1,tyler_haws,session_with_tyler_haws_gym,2,freethrow002_2d.avi,153,204,247,2.550000,3.400000,4.116667
2,tyler_haws,session_with_tyler_haws_gym,3,freethrow003.avi,163,209,260,2.716667,3.483333,4.333333
3,tyler_haws,session_with_tyler_haws_gym,3,freethrow003_2d.avi,161,209,257,2.683333,3.483333,4.283333
4,tyler_haws,session_with_tyler_haws_gym,4,freethrow004.avi,69,123,169,1.150000,2.050000,2.816667


Loaded 101 phase rows across 1 sessions.


,session,clip,path
0,session_with_tyler_haws_gym,2,/Users/kwill55/RVL/personalized-sports-optimiz...
1,session_with_tyler_haws_gym,3,/Users/kwill55/RVL/personalized-sports-optimiz...
2,session_with_tyler_haws_gym,4,/Users/kwill55/RVL/personalized-sports-optimiz...
3,session_with_tyler_haws_gym,5,/Users/kwill55/RVL/personalized-sports-optimiz...
4,session_with_tyler_haws_gym,6,/Users/kwill55/RVL/personalized-sports-optimiz...


Found 138 angle files across 1 sessions.


,athlete,session,clip,file,frame,time_s,angle,value
0,tyler_haws,session_with_tyler_haws_gym,2,freethrow002_angles.csv,0,0.000000,elbow_flex_l,90.046478
1,tyler_haws,session_with_tyler_haws_gym,2,freethrow002_angles.csv,1,0.016667,elbow_flex_l,87.445587
2,tyler_haws,session_with_tyler_haws_gym,2,freethrow002_angles.csv,2,0.033333,elbow_flex_l,85.065498
3,tyler_haws,session_with_tyler_haws_gym,2,freethrow002_angles.csv,3,0.050000,elbow_flex_l,82.990974
4,tyler_haws,session_with_tyler_haws_gym,2,freethrow002_angles.csv,4,0.066667,elbow_flex_l,81.251549


Loaded 365810 angle rows across 1 sessions.


Set up above, plotting below

In [14]:
# get release rows for however many sessions you want 

def get_release_rows(angles_long: pd.DataFrame,
                     phases: pd.DataFrame,
                     sessions: str | list[str] = "all") -> pd.DataFrame:
    merged = angles_long.merge(
        phases[['athlete','session','file','release_frame']],
        on=['athlete','session','file'],
        how='left',
        validate='many_to_one'
    )

    # enforce numeric
    merged['frame'] = pd.to_numeric(merged['frame'], errors='coerce').astype('Int64')

    # filter by sessions
    if sessions != "all":
        if isinstance(sessions, str):
            sessions = [sessions]
        merged = merged[merged['session'].isin(sessions)]

    # filter to release rows
    rel = merged[
        merged['frame'].notna() & (merged['frame'] == merged['release_frame'])
    ].copy()

    return rel


# single/multiple sessions
# release_rows = get_release_rows(angles_long_df, phases_df, sessions=["session_01","session_02"])

# all sessions
# release_rows = get_release_rows(angles_long_df, phases_df, sessions="all")


In [15]:
# plots angles averaged of all free throws over frames

import plotly.express as px

def plot_session_angles_mean(angles_long, session_id: str):
    df = angles_long[angles_long['session'] == session_id].copy()
    # mean across throws (files) for each frame and angle
    agg = (
        df.groupby(["angle", "frame"], as_index=False)["value"]
          .mean()
          .rename(columns={"value": "value_mean"})
    )

    fig = px.line(
        agg,
        x="frame", y="value_mean",
        color="angle",
        hover_data=["angle"]
    )
    fig.update_layout(
        title=f"All Angles over Time — {session_id} (mean across throws)",
        xaxis_title="Frame",
        yaxis_title="Angle (deg)",
        legend_title="Angle",
        hovermode="x unified"
    )
    return fig

# Example:
SESSION = "session_02"
fig = plot_session_angles_mean(angles_long_df, SESSION)
fig.show()

In [16]:
# plot release angles for each free throw in a session 

def _ensure_clip(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure a numeric 'clip' column exists; derive from 'file' if needed."""
    if 'clip' not in df.columns:
        if 'file' in df.columns:
            clip = df['file'].astype(str).str.extract(r'(\d+)')[0]
            df['clip'] = pd.to_numeric(clip, errors='coerce')
        else:
            raise KeyError("release_rows needs 'clip' or 'file' to sort throws.")
    return df

def plot_release_angles_by_throw(release_rows: pd.DataFrame, session_id: str):
    """
    Plot release-time angle values for ALL angles across throws in one session.
    X: throw (clip)
    Y: angle value at release
    """
    df = release_rows[release_rows['session'] == session_id].copy()
    if df.empty:
        raise ValueError(f"No release rows found for {session_id}")

    df = _ensure_clip(df)

    # Order throws numerically (clip 1,2,3,...) and make x a categorical with that order
    df = df.sort_values('clip')
    ordered_clips = df['clip'].dropna().astype(int).unique().tolist()
    df['clip_cat'] = pd.Categorical(df['clip'].astype('Int64').astype(str), categories=[str(c) for c in ordered_clips], ordered=True)

    fig = px.line(
        df,
        x="clip_cat", y="value",
        color="angle",
        markers=True,
        hover_data=["file", "angle", "value", "clip"],
        labels={"clip_cat": "Throw (clip id)", "value": "Angle (deg)", "angle": "Angle"},
        title=f"Release Angle per Throw — {session_id}"
    )
    fig.update_layout(hovermode="x unified", legend_title_text="Angle")
    return fig

# --- Usage (replaces your matplotlib block) ---
SELECTED_ANGLE = "elbow_flex_r"  # not needed for this all-angles plot, but keep if you still use it elsewhere
SESSION = "session_04"

release_rows = get_release_rows(angles_long_df, phases_df, sessions=SESSION)
fig = plot_release_angles_by_throw(release_rows, session_id=SESSION)
fig.show()


KeyError: "['release_frame'] not in index"